# SNDK range asymmetry and inverse ETF study

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config_sndk_inverse.json").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.sndk_inverse_pipeline import run_study

config = json.loads((ROOT / "config_sndk_inverse.json").read_text())
study = run_study(ROOT)  # 核验并复用快照；没有 ETF 快照时下载。不会静默刷新旧数据。
study["data_audit"]

## Daily range definition

In [ ]:
ranges = study["ranges"].copy()
primary = ranges.loc[(ranges.classification == "close_to_close") & ranges.direction.isin(["down", "up"]),
                     ["window", "direction", "days", "mean_range", "median_range", "mean_abs_close_return"]].copy()
percent_columns = ["mean_range", "median_range", "mean_abs_close_return"]
primary[percent_columns] *= 100  # 此表中这三列的单位改成百分比
primary.round(3)

In [ ]:
intervals = study["intervals"].copy()
for column in ["down_minus_up_mean_range", "ci_low", "ci_high"]:
    intervals[column] *= 100  # 百分点，不是百分比变化率
intervals.round(3)

## Prespecified candidate thresholds

In [ ]:
selection = study["selection"]
selection

In [ ]:
metrics = study["metrics"]
columns = ["rule_id", "holding_sessions", "trades", "win_rate", "payoff_ratio",
           "mean_net_return", "cumulative_return", "maximum_drawdown", "sample_status"]
# 完整训练表，包含零交易和亏损规则。下面收益数值仍是小数，例如 0.05 表示 5%。
metrics.loc[(metrics.ticker == "SNDQ") & (metrics.segment == "train"), columns].round(4)

## Win rate, payoff and net returns

In [ ]:
chosen = selection["selected"]
if chosen:
    comparison = metrics.loc[(metrics.rule_id == chosen["rule_id"]) &
                             (metrics.holding_sessions == chosen["holding_sessions"]),
                            ["segment", "ticker", *columns]].copy()
else:
    comparison = metrics.loc[(metrics.rule_id == "unconditional") &
                             (metrics.holding_sessions == 1), ["segment", "ticker", *columns]].copy()
comparison.round(4)

## Matched ETF comparison

In [ ]:
costs = pd.read_csv(ROOT / "reports/sndk_inverse/cost_sensitivity.csv")
if chosen:
    cost_table = costs.loc[(costs.ticker == "SNDQ") & (costs.rule_id == chosen["rule_id"]) &
                          (costs.holding_sessions == chosen["holding_sessions"]),
                          ["segment", "cost_bps", "trades", "win_rate", "mean_net_return",
                           "cumulative_return", "maximum_drawdown"]]
else:
    cost_table = costs.loc[(costs.ticker == "SNDQ") & (costs.rule_id == "unconditional") &
                          (costs.holding_sessions == 1)]
cost_table.round(4)